# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS

Notebook này chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Factual Correctness** (*Độ chính xác nội dung*): so khớp sự kiện với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng từ chối đúng ca ngoài phạm vi / thiếu bằng chứng.

Sau khi hoàn tất, kết quả chi tiết sẽ được xuất tự động ra **`evaluation/results/ragas_evaluation_report.md`**.

In [ ]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env')
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))

In [ ]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_JUDGE_MODEL,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.ingestion.manifest import load_build_manifest

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
JUDGE_MODEL = DEFAULT_JUDGE_MODEL
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
MAX_CASES = None  # Đặt số nguyên (vd: 4) để chạy thử nhanh; None để chạy toàn bộ bộ dữ liệu.

active_manifest_path = PROJECT_ROOT / 'data/knowledge_builds/active/manifest.json'
kb_build_id = load_build_manifest(active_manifest_path).get('build_id') if active_manifest_path.exists() else None
print('Active KB Build ID:', kb_build_id)
print('Judge LLM:', JUDGE_MODEL, '| Embedding:', EMBEDDING_MODEL)

In [ ]:
# 3. Nạp và xem tổng quan bộ câu hỏi đánh giá
dataset_meta, cases = load_evaluation_dataset(DATASET_PATH)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

df_cases = pd.DataFrame([
    {'case_id': c.case_id, 'category': c.category, 'expected': c.expected_behavior, 'question': c.user_input}
    for c in cases
])
print(f'Tổng số câu hỏi sẽ chạy: {len(cases)}')
display(df_cases)

In [ ]:
# 4. Chạy hệ thống Acne Advisor AI trên bộ câu hỏi (tắt cache để đo thực tế)
system_records = await run_system_on_dataset(cases)
df_sys = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'action': r['action'],
        'contexts': len(r['retrieved_contexts']),
        'sources': len(r['sources']),
        'latency_ms': r['latency_ms'],
        'error': r['error'],
    }
    for r in system_records
])
display(df_sys)

In [ ]:
# 5. Chấm điểm bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    embedding_model=EMBEDDING_MODEL,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=JUDGE_MODEL,
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
print('Safe Abstention:', summary['safe_abstention'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

In [ ]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md) và JSON thô
report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)